In [ ]:
import requests
import time
import json
import re
import csv
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import ollama

BASE_URL = "https://br.fas.gov.ru"
START_PAGE = 1
END_PAGE = 28
DELAY = 1.5

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
}

session = requests.Session()
session.headers.update(HEADERS)

def get_page(url):
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response

def is_cartel_case(card):
    card_text = card.get_text().lower()
    indicators = [
        'пункт 2 части 1 статьи 11',
        'п. 2 ч. 1 ст. 11',
        'антиконкурентное соглашение',
        'поддержание цен на торгах',
        'сговор на торгах',
        'признать нарушившими пункт 2',
        'поддержанию цен на торгах'
    ]
    for ind in indicators:
        if ind in card_text:
            return True
    return False

def extract_document_links(page_url):
    print(f"  Loading: {page_url}")
    response = get_page(page_url)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    documents = []
    
    cards = soup.select('div.grey-card')
    for card in cards:
        file_icon = card.select_one('span.glyphicon-file')
        if file_icon is None:
            continue
        
        parent = file_icon.parent
        if parent is None:
            continue
        
        link_elem = parent.select_one('a[href*="/to/"], a[href*="/ca/"]')
        if link_elem is None:
            continue
        
        doc_url = urljoin(BASE_URL, link_elem.get('href', ''))
        card_text = card.get_text().lower()
        
        if re.search(r'(ООО|ЗАО|ОАО|АО)\s*[«"]\s*[Кк]артель', card_text):
            print(f"    Skipped (organisation named 'Cartel'): {doc_url}")
            continue
        
        if not is_cartel_case(card):
            print(f"    Skipped (not a cartel): {doc_url}")
            continue
        
        documents.append(doc_url)
    
    return documents

def extract_text_from_document(doc_url):
    response = get_page(doc_url)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    text_container = soup.select_one('div#document_text_container')
    
    if text_container:
        for unwanted in text_container.select('script, style'):
            unwanted.decompose()
        text = text_container.get_text(separator='\n', strip=True)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r' +', ' ', text)
        return text
    
    return None

def parse_llm_json(text):
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    json_str = json_match.group()
    
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)
    
    data = json.loads(json_str)
    
    if 'auctions' in data:
        clean_auctions = []
        seen = set()
        for a in data['auctions']:
            found = re.findall(r'\b(\d{19}|\d{11})\b', str(a))
            for num in found:
                if num not in seen:
                    seen.add(num)
                    clean_auctions.append(num)
        data['auctions'] = clean_auctions
    
    if 'violators' in data:
        for v in data['violators']:
            if 'inn' not in v or not v['inn']:
                inn_match = re.search(r'\b(\d{10}|\d{12})\b', str(v))
                if inn_match:
                    v['inn'] = inn_match.group(1)
                else:
                    v['inn'] = ""
    
    return data

def extract_with_llm(text, doc_url):
    max_chars = 90000
    if len(text) > max_chars:
        lines = text.split('\n')
        number_lines = []
        for l in lines:
            if re.search(r'\b\d{11,19}\b', l):
                number_lines.append(l)
        text_for_llm = text[:20000] + "\n\n...\n\n" + "\n".join(number_lines[:200]) + "\n\n...\n\n" + text[-15000:]
    else:
        text_for_llm = text

    prompt = f"""
Ты — эксперт-аналитик Федеральной антимонопольной службы. Твоя задача — прочитать текст решения ФАС по делу о картельном сговоре на торгах и выполнить два действия:

1. Найти ВСЕ хозяйствующие субъекты (юридические лица и ИП), признанные виновными в заключении картеля.
2. Найти и извлечь ВСЕ номера закупок (извещений, аукционов, конкурсов), которые в данном решении признаны проведёнными с нарушением антимонопольного законодательства.

**ПРИНЦИПИАЛЬНО ВАЖНОЕ ПРАВИЛО:**
Ты должен извлечь ТОЛЬКО номера формата:
- 19 цифр подряд (закупки по 44-ФЗ)
- 11 цифр подряд (закупки по 223-ФЗ)

**ИГНОРИРУЙ:**
- Номера торговых площадок (3140..., 3150...)
- Номера с буквами и символами (14/ЗК-004159)
- Ссылки и URL
- Любые другие форматы, не соответствующие 19 или 11 цифрам

**КАК ОПРЕДЕЛИТЬ, ЧТО НОМЕР ОТНОСИТСЯ К КАРТЕЛЮ:**

Ты должен проанализировать смысл текста вокруг каждого номера, как это сделал бы живой эксперт. Задай себе следующие вопросы:

1. **Контекст упоминания номера.**
   - Описывается ли рядом с номером конкретное нарушение? (имитация конкуренции, отказ от борьбы, минимальное снижение цены, совпадение IP-адресов, единая инфраструктура)
   - Находится ли номер в разделе, где описываются результаты анализа торгов?
   - Упоминается ли номер в резолютивной части решения (после слова "РЕШИЛА")?

2. **Что НЕ является признаком картеля?**
   - Если рядом с номером есть фразы: «отошёл от сценария», «снижение составило 20-60%», «реальная конкурентная борьба», «не подтвердилось», «не является доказательством».
   - Если номер упоминается в контексте деятельности компаний, НЕ являющихся ответчиками по данному делу.

3. **Структурные подсказки.**
   - Если номер находится в длинном перечислении других номеров, и из контекста понятно, что это список торгов, на которых был реализован сговор — включай ВСЕ номера из этого перечисления.
   - Если номер упоминается как пример нормальной конкуренции — не включай.

**НАРУШИТЕЛИ:**
- Найди в тексте (обычно в разделе "РЕШИЛА") полный перечень лиц, признанных виновными в нарушении пунктов 1-3 части 1 статьи 11 Закона о защите конкуренции.
- Извлеки их полные официальные наименования и ИНН (10 или 12 цифр).

**КРИТИЧЕСКИ ВАЖНО — ТРЕБОВАНИЯ К ОТВЕТУ:**
1. Ответ должен быть ВАЛИДНЫМ JSON и НИЧЕГО БОЛЬШЕ.
2. Убедись, что все строки закрыты кавычками.
3. Убедись, что все {{ и }} сбалансированы.
4. Убедись, что все [ и ] сбалансированы.
5. Не используй комментарии в JSON.
6. Экранируй кавычки внутри строк: "ООО \"Ромашка\""
7. Не ставь запятые после последнего элемента в массиве.
8. Ответ должен начинаться с {{ и заканчиваться }}

**ФОРМАТ ОТВЕТА — СТРОГО JSON:**
{{
  "violators": [
    {{"name": "ООО \\"Название\\"", "inn": "1234567890"}}
  ],
  "auctions": ["0123456789012345678", "12345678901"]
}}

**ТЕКСТ ДЛЯ АНАЛИЗА:**
{text_for_llm}
"""
    
    response = ollama.chat(
        model='deepseek-v3.1:671b-cloud',
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'temperature': 0.0,
            'num_predict': 32768,
            'top_p': 0.95,
            'top_k': 40
        }
    )
    
    result_text = response['message']['content'].strip()
    data = parse_llm_json(result_text)
    
    return data

def build_page_url(page_num):
    params = {
        'type': '1',
        'text': 'картель',
        'category': ['46091', '46334'],
        'page': str(page_num)
    }
    
    query_parts = []
    for key, value in params.items():
        if isinstance(value, list):
            for v in value:
                query_parts.append(f"{key}={v}")
        else:
            query_parts.append(f"{key}={value}")
    
    return f"{BASE_URL}/?{'&'.join(query_parts)}"

def main():
    all_results = []
    
    for page_num in range(START_PAGE, END_PAGE + 1):
        print(f"Page {page_num} of {END_PAGE}")
        
        page_url = build_page_url(page_num)
        doc_urls = extract_document_links(page_url)
        print(f"Cartel cases found: {len(doc_urls)}")
        
        idx = 1
        for doc_url in doc_urls:
            print(f"\n  [{idx}/{len(doc_urls)}] {doc_url}")
            
            text = extract_text_from_document(doc_url)
            if text is None:
                print(f"    Failed to extract text")
                idx = idx + 1
                continue
            
            print(f"    Text: {len(text)} chars")
            print(f"    LLM analysis")
            
            data = extract_with_llm(text, doc_url)
            
            result = {
                'page': page_num,
                'url': doc_url,
                'violators': data.get('violators', []),
                'auctions': data.get('auctions', [])
            }
            all_results.append(result)
            
            violators_count = len(result['violators'])
            auctions_count = len(result['auctions'])
            print(f"    Violators: {violators_count} | Auctions: {auctions_count}")
            if violators_count > 0:
                v_count = 0
                for v in result['violators']:
                    if v_count < 3:
                        print(f"      - {v.get('name', 'N/A')} (INN: {v.get('inn', 'N/A')})")
                        v_count = v_count + 1
                if violators_count > 3:
                    print(f"      ... and {violators_count - 3} more")
            
            time.sleep(DELAY)
            idx = idx + 1
        
        with open(f'fas_page_{page_num}.json', 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
        print(f"  Saved to fas_page_{page_num}.json")
    
    with open('fas_all_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    
    with open('fas_results_ml.csv', 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(['case_url', 'company_name', 'company_inn', 'auction_number'])
        for r in all_results:
            if not r['violators']:
                for auction in r['auctions']:
                    writer.writerow([r['url'], '', '', auction])
            else:
                for v in r['violators']:
                    if not r['auctions']:
                        writer.writerow([r['url'], v.get('name', ''), v.get('inn', ''), ''])
                    else:
                        for auction in r['auctions']:
                            writer.writerow([r['url'], v.get('name', ''), v.get('inn', ''), auction])
    
    with open('fas_results_grouped.csv', 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(['Страница', 'URL', 'Названия компаний', 'ИНН', 'Номера закупок'])
        for r in all_results:
            company_names = ''
            company_inns = ''
            for v in r['violators']:
                if company_names != '':
                    company_names = company_names + '; '
                    company_inns = company_inns + '; '
                company_names = company_names + v.get('name', '')
                company_inns = company_inns + v.get('inn', '')
            
            auctions = ''
            for a in r['auctions']:
                if auctions != '':
                    auctions = auctions + '; '
                auctions = auctions + a
            
            writer.writerow([r['page'], r['url'], company_names, company_inns, auctions])
    
    print(f"Documents processed: {len(all_results)}")

main()

In [ ]:
import requests
import pandas as pd
import json
import time

API_KEY_1 = "YOUR_API_KEY"
API_KEY_2 = "YOUR_API_KEY"

BASE_URL = "https://newapi.clearspending.ru/csinternalapi/v1/filtered-contracts/"
PAGE_SIZE = 50

CURRENT_KEY = 0
API_KEYS = [API_KEY_1, API_KEY_2]

def get_current_key():
    return API_KEYS[CURRENT_KEY]

def switch_key():
    global CURRENT_KEY
    CURRENT_KEY += 1
    return get_current_key()

def get_contracts_by_inn(inn):
    contracts = []
    page = 1
    
    while True:
        for attempt in range(5):
            try:
                params = {
                    "supplier_inns": inn,
                    "page": page,
                    "page_size": PAGE_SIZE,
                    "apikey": get_current_key()
                }
                
                response = requests.get(BASE_URL, params=params, timeout=30)
                
                if response.status_code == 429:
                    wait = (attempt + 1) * 15
                    print(f"    Rate limit, waiting {wait}s")
                    time.sleep(wait)
                    continue
                
                if response.status_code == 403:
                    print(f"    API key exhausted, switching")
                    if switch_key() is None:
                        return contracts
                    continue
                
                data = response.json()
                break
                
            except Exception as e:
                print(f"    Error: {e}, attempt {attempt+1}")
                time.sleep(10)
        else:
            return contracts
        
        if page == 1:
            total_pages = data.get("total_pages", 1)
            if data.get("count", 0) == 0:
                return contracts
        
        page_contracts = data.get("data", [])
        if not page_contracts:
            break
        
        for c in page_contracts:
            c['inn_searched'] = inn
        
        contracts.extend(page_contracts)
        
        if page >= total_pages:
            break
        
        page += 1
        time.sleep(1)
    
    return contracts

with open("финал.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

all_inns = []
for item in data:
    for v in item.get("violators", []):
        inn = v.get("inn")
        if inn:
            all_inns.append(inn)

unique_inns = list(set(all_inns))

print(f"Unique INNs: {len(unique_inns)}")

all_contracts = []

for i, inn in enumerate(unique_inns, 1):
    print(f"[{i}/{len(unique_inns)}] INN: {inn}")
    
    result = get_contracts_by_inn(inn)
    
    if result:
        all_contracts.extend(result)
        print(f"  Contracts: {len(result)}")
    else:
        print(f"  No data")
    
    time.sleep(2)
    
    if i % 100 == 0:
        df_temp = pd.DataFrame(all_contracts)
        df_temp.to_excel(f"checkpoint_contracts_{i}.xlsx", index=False, engine='openpyxl')
        print(f"  Checkpoint saved")

if all_contracts:
    df = pd.DataFrame(all_contracts)
    
    with pd.ExcelWriter("contracts.xlsx", engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name='Contracts')
        
        workbook = writer.book
        worksheet = writer.sheets['Contracts']
        text_format = workbook.add_format({'num_format': '@'})
        
        for col in df.columns:
            if 'inn' in col.lower():
                col_idx = df.columns.get_loc(col)
                worksheet.set_column(col_idx, col_idx, None, text_format)
    
    print(f"Saved {len(all_contracts)} contracts")

In [ ]:
import requests
import pandas as pd
import time

API_KEYS = ["YOUR_API_KEY", "YOUR_API_KEY"]
CURRENT_KEY = 0

BASE_URL = "https://newapi.clearspending.ru/csinternalapi/v1/suppliers/"

def get_current_key():
    return API_KEYS[CURRENT_KEY]

def switch_key():
    global CURRENT_KEY
    CURRENT_KEY += 1
    return get_current_key()

def get_supplier_by_inn(inn):
    for attempt in range(5):
        try:
            url = f"{BASE_URL}{inn}"
            params = {"apikey": get_current_key()}
            
            response = requests.get(url, params=params, timeout=60)
            
            if response.status_code == 429:
                wait = (attempt + 1) * 15
                print(f"    Rate limit, waiting {wait}s")
                time.sleep(wait)
                continue
            
            if response.status_code == 403:
                print(f"    API key exhausted, switching")
                switch_key()
                continue
            
            if response.status_code == 404:
                return None
            
            data = response.json()
            return data
            
        except Exception as e:
            print(f"    Error: {e}, attempt {attempt+1}")
            time.sleep(10)
    
    return None

df_contracts = pd.read_excel("contracts_merged_fixed_inn.xlsx", dtype={'inn_searched': str})
unique_inns = df_contracts['inn_searched'].dropna().unique()

print(f"Unique INNs: {len(unique_inns)}")

suppliers = []

for i, inn in enumerate(unique_inns, 1):
    print(f"[{i}/{len(unique_inns)}] INN: {inn}")
    
    supplier = get_supplier_by_inn(inn)
    
    if supplier:
        supplier['inn_searched'] = inn
        suppliers.append(supplier)
        print(f"  Found: {supplier.get('full_name', 'No name')[:50]}")
    else:
        print(f"  Not found")
    
    time.sleep(2)
    
    if i % 200 == 0:
        df_temp = pd.DataFrame(suppliers)
        df_temp.to_excel(f"checkpoint_suppliers_{i}.xlsx", index=False, engine='openpyxl')
        print(f"  Checkpoint saved")

if suppliers:
    df = pd.DataFrame(suppliers)
    
    with pd.ExcelWriter("suppliers.xlsx", engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name='Suppliers')
        
        workbook = writer.book
        worksheet = writer.sheets['Suppliers']
        text_format = workbook.add_format({'num_format': '@'})
        
        for col in df.columns:
            if 'inn' in col.lower():
                col_idx = df.columns.get_loc(col)
                worksheet.set_column(col_idx, col_idx, None, text_format)
    
    print(f"Saved {len(suppliers)} suppliers")

In [ ]:
import requests
import pandas as pd
import time

API_KEYS = ["YOUR_API_KEY", "YOUR_API_KEY"]
CURRENT_KEY = 0

BASE_URL = "https://newapi.clearspending.ru/csinternalapi/v1/suppliers/"

def get_current_key():
    return API_KEYS[CURRENT_KEY]

def switch_key():
    global CURRENT_KEY
    CURRENT_KEY += 1
    return get_current_key()

def get_is_unfair(inn):
    for attempt in range(5):
        try:
            url = f"{BASE_URL}{inn}"
            params = {"apikey": get_current_key()}
            
            response = requests.get(url, params=params, timeout=60)
            
            if response.status_code == 429:
                wait = (attempt + 1) * 15
                print(f"    Rate limit, waiting {wait}s")
                time.sleep(wait)
                continue
            
            if response.status_code == 403:
                print(f"    API key exhausted, switching")
                switch_key()
                continue
            
            if response.status_code == 404:
                return None
            
            data = response.json()
            return data.get('is_unfair')
            
        except Exception as e:
            print(f"    Error: {e}, attempt {attempt+1}")
            time.sleep(10)
    
    return None

df = pd.read_excel("all_suppliers_fixed.xlsx", dtype={'inn': str})
unique_inns = df['inn'].dropna().unique()

print(f"Unique INNs: {len(unique_inns)}")

df['is_unfair'] = None

for i, inn in enumerate(unique_inns, 1):
    print(f"[{i}/{len(unique_inns)}] INN: {inn}")
    
    is_unfair = get_is_unfair(inn)
    
    if is_unfair is not None:
        df.loc[df['inn'] == inn, 'is_unfair'] = is_unfair
        print(f"  is_unfair = {is_unfair}")
    else:
        print(f"  Not found")
    
    time.sleep(2)
    
    if i % 100 == 0:
        df.to_excel(f"checkpoint_unfair_{i}.xlsx", index=False, engine='openpyxl')
        print(f"  Checkpoint saved")

with pd.ExcelWriter("suppliers_with_unfair.xlsx", engine='xlsxwriter') as writer:
    df.to_excel(writer, index=False, sheet_name='Suppliers')
    
    workbook = writer.book
    worksheet = writer.sheets['Suppliers']
    text_format = workbook.add_format({'num_format': '@'})
    
    for col in df.columns:
        if 'inn' in col.lower():
            col_idx = df.columns.get_loc(col)
            worksheet.set_column(col_idx, col_idx, None, text_format)

print("Done")

In [ ]:
import pandas as pd

df1 = pd.read_excel("contracts_part1.xlsx")
df2 = pd.read_excel("contracts_part2.xlsx")

merged = pd.concat([df1, df2], ignore_index=True)
merged.to_excel("contracts_merged.xlsx", index=False, engine='openpyxl')

print(f"Merged: {len(merged)} rows")

In [ ]:
import pandas as pd

df = pd.read_excel("contracts_merged.xlsx", dtype={'inn_searched': str})

def fix_inn(inn):
    if pd.isna(inn):
        return None
    inn_str = str(inn).strip().replace('.0', '')
    if not inn_str:
        return None
    if len(inn_str) <= 10:
        return inn_str.zfill(10)
    else:
        return inn_str.zfill(12)

df['inn_searched'] = df['inn_searched'].apply(fix_inn)
df.to_excel("contracts_merged_fixed_inn.xlsx", index=False, engine='openpyxl')

print(f"Saved {len(df)} rows")

In [ ]:
import pandas as pd

df_contracts = pd.read_excel("contracts_merged_fixed_inn.xlsx", dtype={'inn_searched': str})
df_suppliers = pd.read_excel("suppliers_with_is_unfair.xlsx", dtype={'inn': str})

df_merged = df_contracts.merge(
    df_suppliers,
    left_on='inn_searched',
    right_on='inn',
    how='inner'
)

df_merged = df_merged.drop('inn', axis=1)

with pd.ExcelWriter("contracts_with_supplier_data.xlsx", engine='xlsxwriter') as writer:
    df_merged.to_excel(writer, index=False, sheet_name='Contracts')
    
    workbook = writer.book
    worksheet = writer.sheets['Contracts']
    text_format = workbook.add_format({'num_format': '@'})
    
    for col in df_merged.columns:
        if 'inn' in col.lower():
            col_idx = df_merged.columns.get_loc(col)
            worksheet.set_column(col_idx, col_idx, None, text_format)

print(f"Before: {len(df_contracts)}")
print(f"After: {len(df_merged)}")

In [ ]:
import pandas as pd

df = pd.read_excel("contracts_with_supplier_data.xlsx")

before = len(df)
df_filtered = df[df['fz'] != 94]
after = len(df_filtered)

print(f"Removed: {before - after}")

with pd.ExcelWriter("contracts_without_fz94.xlsx", engine='xlsxwriter') as writer:
    df_filtered.to_excel(writer, index=False, sheet_name='Contracts')
    
    workbook = writer.book
    worksheet = writer.sheets['Contracts']
    text_format = workbook.add_format({'num_format': '@'})
    
    for col in df_filtered.columns:
        if 'inn' in col.lower():
            col_idx = df_filtered.columns.get_loc(col)
            worksheet.set_column(col_idx, col_idx, None, text_format)

In [ ]:
import requests
import pandas as pd
import time

API_KEYS = [
    "YOUR_API_KEY",
    "YOUR_API_KEY",
    "YOUR_API_KEY",
    "YOUR_API_KEY",
    "YOUR_API_KEY"
]

CURRENT_KEY = 0

BASE_URL_44 = "https://newapi.clearspending.ru/csinternalapi/v1/contracts44/"
BASE_URL_223 = "https://newapi.clearspending.ru/csinternalapi/v1/contracts223/"

def get_current_key():
    return API_KEYS[CURRENT_KEY]

def switch_key():
    global CURRENT_KEY
    CURRENT_KEY += 1
    return get_current_key()

def get_contract_details(fz, regnum):
    for attempt in range(5):
        try:
            if fz == "44":
                url = f"{BASE_URL_44}{regnum}"
            elif fz == "223":
                url = f"{BASE_URL_223}{regnum}"
            else:
                return None
            
            params = {"apikey": get_current_key()}
            response = requests.get(url, params=params, timeout=75)
            
            if response.status_code == 429:
                wait = (attempt + 1) * 15
                print(f"    Rate limit, waiting {wait}s")
                time.sleep(wait)
                continue
            
            if response.status_code == 403:
                print(f"    API key exhausted, switching")
                switch_key()
                continue
            
            if response.status_code == 404:
                return None
            
            data = response.json()
            data['fz_searched'] = fz
            data['regnum_searched'] = regnum
            return data
            
        except Exception as e:
            print(f"    Error: {e}, attempt {attempt+1}")
            time.sleep(10)
    
    return None

df = pd.read_excel("zakupki_with_collusion.xlsx")
df_filtered = df[df['fz'].astype(str).isin(['44', '223'])]
unique_contracts = df_filtered[['fz', 'regnum']].drop_duplicates()
contracts_list = unique_contracts.to_dict('records')

print(f"Total contracts: {len(contracts_list)}")

contracts_data = []

for i, contract in enumerate(contracts_list, 1):
    fz = str(contract['fz'])
    regnum = str(contract['regnum'])
    
    print(f"[{i}/{len(contracts_list)}] FZ {fz}, regnum: {regnum}")
    
    data = get_contract_details(fz, regnum)
    
    if data:
        contracts_data.append(data)
        print(f"  Success")
    else:
        print(f"  Not found")
    
    time.sleep(0.5)
    
    if i % 500 == 0:
        df_temp = pd.DataFrame(contracts_data)
        df_temp.to_excel(f"checkpoint_details_{i}.xlsx", index=False, engine='openpyxl')
        print(f"  Checkpoint saved")

def flatten_dict(d, parent_key=''):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}_{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key).items())
        elif isinstance(v, list):
            items.append((new_key, str(v)))
        else:
            items.append((new_key, v))
    return dict(items)

if contracts_data:
    flattened = [flatten_dict(d) for d in contracts_data]
    df_result = pd.DataFrame(flattened)
    
    with pd.ExcelWriter("contracts_details.xlsx", engine='xlsxwriter') as writer:
        df_result.to_excel(writer, index=False, sheet_name='Contracts')
        
        workbook = writer.book
        worksheet = writer.sheets['Contracts']
        text_format = workbook.add_format({'num_format': '@'})
        
        for col in df_result.columns:
            if 'regnum' in col.lower() or 'inn' in col.lower():
                col_idx = df_result.columns.get_loc(col)
                worksheet.set_column(col_idx, col_idx, None, text_format)
    
    print(f"Saved {len(contracts_data)} contracts")